In [ ]:
# =============================================================================
# DAY 25 — PPO & RLHF
# From fragile policy gradients → stable clipped updates → aligning LLMs
# =============================================================================
#
# Yesterday (Day 24): REINFORCE / A2C learn π by
#   θ ← θ + α · ∇ log π(a|s) · Advantage
#
# Today:
#   1) Why that still breaks in practice
#   2) PPO's fix: never trust a huge policy change in one update
#   3) GAE: a smarter advantage estimate
#   4) RLHF: how ChatGPT-style models use PPO + human preferences
#
#
# ---------------------------------------------------------------------------
# 1) THE PROBLEM WITH VANILLA POLICY GRADIENTS
# ---------------------------------------------------------------------------
#
# Vanilla PG / REINFORCE / basic A2C use an ON-POLICY update:
#   collect data with π_old  →  take ONE (or a few) gradient steps on that data
#
# Two failure modes:
#
# (A) STEP TOO BIG
#   One noisy high-advantage sample can shove π hard.
#   New π puts almost all mass on a weird action → next rollout is garbage
#   → you cannot recover easily. "Policy collapse."
#
# (B) REUSE DATA TOO MUCH (naive multi-epoch)
#   People want sample efficiency: run many SGD epochs on the same batch.
#   But the PG theorem assumes data came from the CURRENT π.
#   After epoch 1, π has moved. Epoch 5 is updating with STALE probabilities
#   → biased, often explosive updates.
#
# Picture a thermostat:
#   Vanilla PG = "room is cold, crank heat to MAX." Overshoots, then crashes.
#   PPO        = "nudge heat a little; clip the knob so you can't spin it 10×."
#
# TRPO (older sibling) constrained KL(π_new || π_old) with complex second-order
# math. PPO keeps the SAME IDEA (stay close to the old policy) with a simple
# CLIP — first-order SGD, easy to implement, works shockingly well.
#
#
# ---------------------------------------------------------------------------
# 2) PPO: THE CLIPPED SURROGATE OBJECTIVE
# ---------------------------------------------------------------------------
#
# Importance ratio (how much MORE likely is this action under the new policy?):
#
#   r_t(θ) = π_θ(a_t | s_t) / π_θ_old(a_t | s_t)
#
#   r = 1     → new policy agrees with old on this action
#   r = 2     → new policy is 2× more likely to pick this action
#   r = 0.5   → new policy half as likely
#
# Unclipped surrogate (like "reuse data carefully"):
#
#   L^CPI = E[ r_t(θ) · A_t ]
#
#   If A > 0 (good action): maximizing L pushes r UP  → make action more likely
#   If A < 0 (bad action):  maximizing L pushes r DOWN → make action less likely
#
# Problem: r can explode (r = 100) → huge policy jump.
#
# PPO CLIP (ε typically 0.1 or 0.2):
#
#   L^CLIP = E[  min(  r_t · A_t  ,  clip(r_t, 1−ε, 1+ε) · A_t  )  ]
#
# English:
#   "Take the pessimistic (more conservative) of:
#      (1) the normal importance-weighted advantage
#      (2) the advantage if we FAKE that r stayed inside [1−ε, 1+ε]"
#
# What the clip does, case by case:
#
#   Good action (A > 0):
#     Want r to grow, but once r > 1+ε the clipped term stops giving extra
#     reward for growing further. Gradient on that sample ≈ 0 for more increase.
#     → cannot over-boost a lucky action in one update.
#
#   Bad action (A < 0):
#     Want r to shrink, but once r < 1−ε further decrease is not rewarded
#     by the min(...). → cannot over-kill / collapse probability in one step.
#
# Mental model: a seatbelt on the policy update. You may improve, but only
# within a trust region of about ±ε relative probability change per batch.
#
#
# ---------------------------------------------------------------------------
# 3) FULL PPO LOSS
# ---------------------------------------------------------------------------
#
# Real PPO optimizes three pieces together:
#
#   L(θ) = E[
#            L^CLIP          # policy improvement (clipped)
#          − c1 · L^VF       # value / critic fit
#          + c2 · S[π](s)    # entropy bonus (explore)
#          ]
#
# (1) Policy term L^CLIP
#     As above. Often written with a sign flip if you MINIMIZE loss in code:
#       policy_loss = − L^CLIP
#
# (2) Value term L^VF
#     Critic V_φ(s) predicts returns (or discounted rewards-to-go).
#     L^VF = ( V_φ(s_t) − R_t )²
#     Sometimes also clipped value updates (optional, same spirit as policy clip).
#
# (3) Entropy bonus S[π]
#     H = −Σ_a π(a|s) log π(a|s)   (or differential entropy for Gaussians)
#     Encourages π to stay a bit random → better exploration early.
#     c2 is small (e.g. 0.01). Too big → never commits; too small → may collapse.
#
# Typical coefficients: c1 ≈ 0.5, c2 ≈ 0.01, ε ∈ {0.1, 0.2}.
#
# In code you often see:
#   loss = policy_loss + c1 * value_loss − c2 * entropy
#   (signs depend on whether policy_loss already includes the minus)
#
#
# ---------------------------------------------------------------------------
# 4) GENERALIZED ADVANTAGE ESTIMATION (GAE)
# ---------------------------------------------------------------------------
#
# Advantage A(s,a) = "how much better was this action than average here?"
#
# Extreme estimators:
#
#   Monte Carlo:  A_t = G_t − V(s_t)
#     G_t = full return to end of episode
#     Low bias, HIGH variance (one lucky trajectory swamps the signal)
#
#   1-step TD:    A_t ≈ δ_t = r_t + γ V(s_{t+1}) − V(s_t)
#     Low variance, HIGHER bias (trusts V, which may be wrong)
#
# GAE blends them with λ ∈ [0, 1]:
#
#   δ_t = r_t + γ V(s_{t+1}) − V(s_t)          # TD residual
#
#   A_t^GAE(γ,λ) = Σ_{l=0}^{∞} (γλ)^l · δ_{t+l}
#
#   λ = 0  → pure 1-step TD (A_t = δ_t)
#   λ = 1  → essentially Monte Carlo (sums all future δ's → G − V)
#   λ ≈ 0.95 → sweet spot used in most PPO papers / codebases
#
# Intuition:
#   λ is a bias–variance dial.
#   Slightly trust the critic for near-term bootstraps, but still look
#   several steps ahead so credit assignment is not myopic.
#
# PPO almost always uses GAE for Â_t inside L^CLIP.
#
#
# ---------------------------------------------------------------------------
# 5) PPO ALGORITHM (SIMPLIFIED)
# ---------------------------------------------------------------------------
#
# Repeat forever:
#
#   1. ROLLOUT
#      Using current policy π_θ (call it π_old after freeze):
#        interact with env for T steps (or N parallel envs × T)
#        store: s_t, a_t, r_t, done_t, log π_old(a_t|s_t), V(s_t)
#
#   2. ADVANTAGES & RETURNS
#      Compute δ_t, then GAE advantages Â_t
#      Returns R_t = Â_t + V(s_t)   (targets for the critic)
#      Often normalize Â across the batch (mean 0, std 1)
#
#   3. OPTIMIZE (K epochs over minibatches of that rollout)
#      For each minibatch:
#        r_t = exp( log π_θ(a|s) − log π_old(a|s) )
#        L_clip, L_vf, entropy → total loss
#        Adam step on θ (and critic params)
#      STOP early if mean KL(π_θ || π_old) gets too large (optional safety)
#
#   4. SET π_old ← π_θ  and go back to step 1
#
# Why multiple epochs are OK now:
#   The clip (and optional KL early-stop) prevent the multi-epoch reuse from
#   walking π arbitrarily far from the data-collecting policy.
#
# Practical knobs:
#   rollout length T, num envs, minibatch size, K epochs,
#   ε, γ, λ, c1, c2, learning rate (+ often linear LR decay)
#
#
# ---------------------------------------------------------------------------
# 6) RLHF: THE FULL PIPELINE
#    Reinforcement Learning from Human Feedback
# ---------------------------------------------------------------------------
#
# Goal: make a language model answer in ways HUMANS prefer
# (helpful, honest, harmless) — not just "likely under the pretraining data."
#
# Pretrained LLM next-token prediction ≠ aligned assistant.
# RLHF = a recipe to steer the model with preference data + PPO.
#
# Below: stages as commonly described for InstructGPT / ChatGPT-style systems.
# Names in your outline (SFT, RM, PA, ACTOR, KL, PPO, Eval) map onto this flow.
#
#
# --- Stage 1: Supervised Fine-Tuning (SFT) ---------------------------------
#
# Start from a pretrained base model (next-token LM).
# Collect demonstration data: prompts + high-quality target answers
# (labelers write good replies, or curated examples).
#
# Train with ordinary supervised loss:
#   maximize log P_θ(answer | prompt)   (teacher forcing)
#
# Output: π_SFT — a model that already "sounds like an assistant."
# This is the warm start for everything later (and often the KL anchor).
#
#
# --- Stage 2: Reward Modeling (RM) -----------------------------------------
#
# Humans (or stronger models) compare answers:
#   Given prompt x, sample two replies (y_w, y_l) = (preferred, rejected)
#
# Train a reward model r_φ(x, y) — usually another LM with a scalar head —
# so preferred answers score higher:
#
#   L_RM = − E[ log σ( r_φ(x, y_w) − r_φ(x, y_l) ) ]
#
# English: "probability that the preferred answer beats the rejected one."
#
# Output: r_φ — a learned stand-in for human taste. Now we can score ANY
# new completion without calling a human every time.
#
#
# --- Stage 3: Preference Alignment (PA) ------------------------------------
#
# Broad name for "use preference data to align the policy."
# Classic RLHF does this via reward model + RL (stages 4–6).
#
# Alternatives in modern stacks (not mutually exclusive):
#   - PPO-RLHF (this notebook's focus)
#   - DPO / IPO / KTO: skip explicit RM+PPO; directly prefer y_w over y_l
#     with a closed-form loss on the policy
#
# Think of PA as the PRODUCT GOAL; RM+PPO is one ALGORITHM to get there.
#
#
# --- Stage 4: Actor Training (ACTOR) ---------------------------------------
#
# The ACTOR is the policy we will RL-train: usually initialize
#   π_θ ← π_SFT
#
# In LLM RLHF, an "action" is typically the next token (or a full continuation
# treated as a trajectory of tokens). State ≈ prompt + tokens so far.
#
# Actor generates completions y ~ π_θ(· | x) that the reward model will score.
#
#
# --- Stage 5: KL Penalty ---------------------------------------------------
#
# Purely maximizing r_φ is dangerous:
#   Model finds adversarial gibberish that fools the reward model ("reward hacking")
#   Or drifts so far from natural language that fluency dies.
#
# Fix: penalize divergence from a reference policy π_ref (usually π_SFT):
#
#   reward_used = r_φ(x, y) − β · KL( π_θ(·|x) || π_ref(·|x) )
#
#   β = KL coefficient (temperature of "how hard we allow drift")
#
# Intuition: "Get higher reward, but stay close to the SFT model that speaks
# proper English and already behaves reasonably."
#
# This KL term is the LLM cousin of PPO's trust region — both say
# "don't change the policy too wildly."
#
#
# --- Stage 6: PPO Training -------------------------------------------------
#
# Now run PPO where:
#   - environment "state"  = conversation / prompt context
#   - "action"             = generated tokens
#   - "reward"             = RM score − β·KL (often assigned at EOS / sequence end)
#   - policy               = actor LLM π_θ
#   - critic               = value head (or separate value network) on same backbone
#
# Loop:
#   sample prompts → generate completions with π_θ
#   score with r_φ and KL against π_ref
#   compute GAE advantages
#   PPO clipped updates on actor (+ value loss, entropy if used)
#
# Result: π_RLHF — prefers high-reward answers without straying too far from SFT.
#
#
# --- Stage 7: Evaluation ---------------------------------------------------
#
# Automatic metrics are not enough (BLEU ≠ "good assistant").
# Typical eval:
#   - Human preference win rate vs SFT / vs previous checkpoint
#   - Held-out RM scores (careful: over-optimizing the same RM leaks)
#   - Safety / red-team suites (toxicity, jailbreaks)
#   - Instruction-following benchmarks (MMLU, MT-Bench, Arena-style pairwise)
#   - Regression checks: didn't we break coding / math while aligning?
#
# Only after eval do you ship (or reject and retune β, data mix, RM quality).
#
#
# ---------------------------------------------------------------------------
# BIG PICTURE
# ---------------------------------------------------------------------------
#
#   Vanilla PG ──(too big steps / stale data)──► unstable π
#        │
#        ▼
#   PPO clip + GAE ──► stable on-policy RL (games agents, robots, …)
#        │
#        ▼
#   RLHF = SFT warm-start + preference RM + KL-to-SFT + PPO on the LLM actor
#        │
#        ▼
#   Evaluated, aligned assistant policy
#
# Remember:
#   PPO = "how to update π safely"
#   GAE = "how to estimate advantage with less noise"
#   RLHF = "what reward signal to optimize for human preferences" + PPO machinery


In [ ]:
# Part 1: PPO on CartPole — Setup
# ================================
# Goal of Part 1: implement PPO (clipped surrogate + GAE) on CartPole-v1,
# using the ideas from the concept cell above.
#
# What each import is for:
#   numpy / torch          → arrays, tensors, autograd
#   nn, F, optim           → ActorCritic net, activations, Adam
#   Categorical            → discrete policy π(a|s) = softmax(logits)
#   gymnasium as gym       → CartPole environment (modern OpenAI Gym fork)
#   matplotlib             → learning curves
#   deque                  → optional rolling window of recent returns
#
# Use the project .venv kernel (gymnasium + torch). Classic `import gym`
# often fails with ModuleNotFoundError on a stock system Python.

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym  # not `import gym` — same make()/step() API for CartPole
import matplotlib.pyplot as plt
from torch.distributions import Categorical
from collections import deque

# Reproducibility: same seeds → closer-to-same weight inits / action samples
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"gymnasium {gym.__version__}")


In [ ]:
# Actor–Critic Network (shared trunk)
# -----------------------------------
# PPO needs TWO heads on the same state features:
#
#   ACTOR  (policy)  → logits over actions → Categorical π_θ(a|s)
#                      used for sampling actions + computing r_t = π/π_old
#   CRITIC (value)   → scalar V_φ(s)
#                      used for GAE advantages Â = GAE(δ) with δ = r+γV′−V
#
# Shared layers: actor and critic see the same representation of the state.
# That saves compute and usually learns faster than two totally separate nets.
#
# Why Tanh + orthogonal init?
#   Tanh keeps hidden activations bounded (common in OpenAI/Spinning Up PPO).
#   Orthogonal init (gain √2 on hidden, small gain on actor) reduces early
#   exploding/vanishing gradients so the first rollouts are not pure chaos.
#   Actor gain 0.01 → start near-uniform π (safe exploration).
#   Critic gain 1.0  → value head can move more freely at the start.

class ActorCritic(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=64):
        super().__init__()
        # Shared body: state_dim → hidden → hidden
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor = nn.Linear(hidden_dim, n_actions)  # one logit per action
        self.critic = nn.Linear(hidden_dim, 1)         # one number: V(s)

        # Orthogonal initialization for stability
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        # Override heads with PPO-typical gains after the loop
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        # x: (batch, state_dim) → logits (batch, n_actions), values (batch,)
        h = self.shared(x)
        return self.actor(h), self.critic(h).squeeze(-1)


In [ ]:
# GAE and Rollout Buffer
# ======================
#
# TOPIC — why this cell exists
# ---------------------------
# PPO needs two things from a batch of experience:
#
#   1) ADVANTAGES Â_t  — "was this action better than average?" (for L^CLIP)
#   2) RETURNS    R_t  — targets for the critic: usually R_t = Â_t + V(s_t)
#
# Generalized Advantage Estimation (GAE) builds Â from TD residuals:
#
#   δ_t = r_t + γ V(s_{t+1}) − V(s_t)          # 1-step TD error
#   Â_t = δ_t + (γλ) δ_{t+1} + (γλ)² δ_{t+2} + ...
#
#   λ ≈ 0.95  → blend: less noisy than full Monte Carlo, less biased than 1-step
#   λ = 0     → Â_t = δ_t only
#   λ = 1     → essentially Monte Carlo (G − V)
#
# When an episode ENDS (done=1), there is no "next state" — we zero the
# bootstrap: multiply next value / next GAE by (1 − done).
#
# RolloutBuffer = a simple list store for one PPO collect phase.
#   During rollout:  .add(...) each step
#   Before update:   .get_tensors(...) → compute GAE, pack torch tensors
#   After update:    .clear() and collect again (on-policy: fresh data only)
#
#
# CODE APPROACH
# -------------
# compute_gae: walk the trajectory BACKWARDS so Â_t can reuse Â_{t+1}:
#   Â_t = δ_t + γ λ (1−done_t) Â_{t+1}
# (same recurrence as the infinite sum above, computed in O(T))
#
# get_tensors: numpy → GAE → torch, then normalize Â (mean 0, std 1)
# so the policy loss scale stays stable across batches.


def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """
    rewards, values, dones: length-T arrays from one rollout
    next_value: V(s_T) bootstrap for the step AFTER the last stored transition
                (0.0 if that step was terminal)

    Returns:
      advantages Â_t  and  returns R_t = Â_t + V(s_t)   (both length T)
    """
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    last_gae = 0.0  # this is Â_{t+1} as we walk backward
    for t in reversed(range(T)):
        # If episode ended at t, do not bootstrap into the next episode
        if t == T - 1:
            next_nonterminal = 1.0 - dones[t]
            next_val = next_value  # critic's V(s_after_last_step)
        else:
            next_nonterminal = 1.0 - dones[t]
            next_val = values[t + 1]

        # TD residual δ_t
        delta = rewards[t] + gamma * next_val * next_nonterminal - values[t]
        # GAE recurrence: Â_t = δ_t + γλ(1−done) Â_{t+1}
        advantages[t] = last_gae = delta + gamma * lam * next_nonterminal * last_gae

    # Critic target: return ≈ advantage + baseline value
    returns = advantages + values
    return advantages, returns


class RolloutBuffer:
    """On-policy storage for one PPO collect → update cycle."""

    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []  # log π_old(a|s) — frozen for the ratio r_t
        self.rewards = []
        self.values = []     # V(s) at collect time (for GAE)
        self.dones = []      # 1.0 if episode ended after this step

    def add(self, state, action, log_prob, reward, value, done):
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def clear(self):
        # Fresh buffer after each PPO update (do not reuse stale on-policy data)
        self.__init__()

    def get_tensors(self, gamma, lam, next_value, device):
        # Pack lists → tensors; compute GAE on CPU numpy then move Â, R to device
        states = torch.tensor(np.array(self.states), dtype=torch.float32, device=device)
        actions = torch.tensor(self.actions, dtype=torch.long, device=device)
        old_log_probs = torch.tensor(self.log_probs, dtype=torch.float32, device=device)
        rewards = np.array(self.rewards, dtype=np.float32)
        values = np.array(self.values, dtype=np.float32)
        dones = np.array(self.dones, dtype=np.float32)

        advantages, returns = compute_gae(rewards, values, dones, next_value, gamma, lam)
        advantages = torch.tensor(advantages, dtype=torch.float32, device=device)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # Normalize advantages over the whole rollout (common PPO trick)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return states, actions, old_log_probs, returns, advantages
